## Imports and Setup

In [15]:
import os
import math
import heapq
import folium
import pandas as pd
import openrouteservice
from dotenv import load_dotenv
from IPython.display import display

print("DONE! Dependencies Included")

DONE! Dependencies Included


##  Load and Preprocess the Data

In [16]:
file_path = r'./us_hospital_locations.csv'
hospital_data = pd.read_csv(file_path)

# Extract needed columns
hospital_locations = hospital_data[['NAME', 'LATITUDE', 'LONGITUDE']].dropna()
hospital_locations.columns = ['Hospital Name', 'Latitude', 'Longitude']

# Show sample hospitals
hospital_locations.head()

,Hospital Name,Latitude,Longitude
0,CENTRAL VALLEY GENERAL HOSPITAL,36.336159,-119.645667
1,LOS ROBLES HOSPITAL & MEDICAL CENTER - EAST CA...,34.154939,-118.815736
2,EAST LOS ANGELES DOCTORS HOSPITAL,34.023647,-118.184165
3,SOUTHERN CALIFORNIA HOSPITAL AT HOLLYWOOD,34.096391,-118.325235
4,KINDRED HOSPITAL BALDWIN PARK,34.063039,-117.967438


## Graph Class and Haversine Function 

In [17]:
# Graph class to build the network
class Graph:
    def __init__(self):
        self.nodes = {}  # {node_id: (latitude, longitude)}
        self.edges = {}  # {node_id: [(neighbor_id, distance)]}

    def add_node(self, node_id, lat, lon):
        self.nodes[node_id] = (lat, lon)
        self.edges[node_id] = []

    def add_edge(self, from_id, to_id, distance):
        self.edges[from_id].append((to_id, distance))
        self.edges[to_id].append((from_id, distance))  # Undirected graph

# Haversine formula to compute distance between two points
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

## Build the Graph (User + Hospitals)

In [18]:
# Initialize the graph
graph = Graph()

# Define your user location (Example: New York City)
user_location = (40.7128, -74.0060)
graph.add_node('user', user_location[0], user_location[1])

# Add all hospitals as nodes and connect to user
for idx, row in hospital_locations.iterrows():
    hospital_id = f"hospital_{idx}"
    graph.add_node(hospital_id, row['Latitude'], row['Longitude'])
    # Edge: user <-> hospital (distance based on Haversine)
    distance = haversine(user_location[0], user_location[1], row['Latitude'], row['Longitude'])
    graph.add_edge('user', hospital_id, distance)

## Dijkstra’s Algorithm (Core Pathfinding)

In [19]:
# Dijkstra’s Algorithm (professional version with comments and edge handling)
def dijkstra(graph, start_node):
    """
    Dijkstra's algorithm to compute shortest paths from start_node to all other nodes.
    Returns:
        - distances: dict {node: shortest distance from start_node}
        - previous_nodes: dict {node: previous node in optimal path}
    """
    # Initialize distances and previous nodes
    distances = {node: float('inf') for node in graph.nodes}
    previous_nodes = {node: None for node in graph.nodes}

    # Distance to start node is 0
    distances[start_node] = 0

    # Priority queue: (distance, node)
    priority_queue = [(0, start_node)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)

        # Skip if we already found a better path
        if current_distance > distances[current_node]:
            continue

        for neighbor, weight in graph.edges[current_node]:
            if weight < 0:
                raise ValueError("Graph contains negative weight edge, which Dijkstra's cannot handle.")

            distance = current_distance + weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous_nodes[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    return distances, previous_nodes

## Find Nearest Hospital

In [20]:
# Function to calculate the nearest hospital based on the user's location
def find_nearest_hospital(user_location, hospital_data):
    distances = []
    for idx, row in hospital_data.iterrows():
        hospital_location = (row['Latitude'], row['Longitude'])
        # Use Haversine formula to calculate initial distance (for sorting purposes)
        distance = haversine(user_location[0], user_location[1], hospital_location[0], hospital_location[1])
        distances.append((row['Hospital Name'], distance, hospital_location))
    
    # Sort by distance and return the nearest one
    nearest = sorted(distances, key=lambda x: x[1])[0]
    return nearest  # (Hospital Name, distance in km, (lat, lon))

In [21]:
# Run Dijkstra from the user node
distances, previous_nodes = dijkstra(graph, 'user')

# Find the nearest hospital (minimum distance)
nearest_hospital = None
min_distance = float('inf')

for node, distance in distances.items():
    if node != 'user' and distance < min_distance:
        min_distance = distance
        nearest_hospital = node

if nearest_hospital:
    hospital_idx = int(nearest_hospital.split('_')[1])
    hospital_name = hospital_locations.iloc[hospital_idx]['Hospital Name']
    hospital_coords = graph.nodes[nearest_hospital]
    print(f"Nearest hospital: {hospital_name}")
    print(f"Distance (straight line): {min_distance:.2f} km")
    print(f"Location: {hospital_coords}")
else:
    print("No hospital found.")

Nearest hospital: NEW YORK-PRESBYTERIAN/LOWER MANHATTAN HOSPITAL
Distance (straight line): 0.26 km
Location: (40.7105167880001, -74.0052978259999)


## Real Driving Route + Map Visualization Inline

In [22]:
# Initialize OpenRouteService Client with your API key
client = openrouteservice.Client(key="5b3ce3597851110001cf62486b9a970bdbb14de28fe65eea52af5978")

def get_best_route(user_location, hospital_location):
    """
    Fetch the optimal driving route using OpenRouteService.
    Returns:
        - route geometry (for plotting)
        - distance in km
        - duration in minutes
    """
    coords = [user_location[::-1], hospital_location[::-1]]  # ORS expects (lon, lat)
    route = client.directions(
        coordinates=coords,
        profile='driving-car',
        format='geojson'
    )
    geometry = route['features'][0]['geometry']
    properties = route['features'][0]['properties']['segments'][0]
    distance_km = properties['distance'] / 1000  # meters to km
    duration_min = properties['duration'] / 60   # seconds to minutes
    return geometry, distance_km, duration_min

def visualize_nearest_hospital_with_real_route(user_location, hospital_location, hospital_name):
    """
    Visualizes the optimal route from user location to the nearest hospital inline in Jupyter.
    """
    route_geometry, route_distance, route_duration = get_best_route(user_location, hospital_location)

    print(f"\nOptimal route to {hospital_name}:")
    print(f"- Distance: {route_distance:.2f} km")
    print(f"- Estimated travel time: {route_duration:.1f} minutes")

    # Create the map
    route_map = folium.Map(location=user_location, zoom_start=13)

    # Add user marker
    folium.Marker(
        location=user_location,
        popup=f"{hospital_name} (User Location)",
        icon=folium.Icon(color='blue')
    ).add_to(route_map)

    # Add hospital marker
    folium.Marker(
        location=hospital_location,
        popup=f"{hospital_name} ({route_distance:.2f} km, {route_duration:.1f} min)",
        icon=folium.Icon(color='red')
    ).add_to(route_map)

    # Add the actual route (polyline of roads)
    folium.GeoJson(route_geometry, name='route').add_to(route_map)

    # Display the map inline
    display(route_map)

# Run visualization with real driving route
visualize_nearest_hospital_with_real_route(user_location, hospital_coords, hospital_name)


Optimal route to NEW YORK-PRESBYTERIAN/LOWER MANHATTAN HOSPITAL:
- Distance: 0.51 km
- Estimated travel time: 2.1 minutes


## Final Test: Running for Multiple Locations

In [23]:
# User locations (5 different public locations in the U.S.)
user_locations = {
    "Times Square, NYC": (40.7580, -73.9855),
    "Central Park, NYC": (40.7851, -73.9683),
    "Golden Gate Bridge, SF": (37.8199, -122.4783),
    "White House, DC": (38.8977, -77.0365),
    "LAX Airport, LA": (33.9416, -118.4085)  # Replacing Hollywood Sign with LAX
}

# Function to visualize route for multiple locations
def visualize_multiple_routes(user_locations, hospital_locations):
    for location_name, user_location in user_locations.items():
        # Find nearest hospital for the current user location
        nearest_hospital = find_nearest_hospital(user_location, hospital_locations)
        hospital_name = nearest_hospital[0]
        hospital_coords = nearest_hospital[2]

        # Get the best route using OpenRouteService
        route_geometry, route_distance, route_duration = get_best_route(user_location, hospital_coords)

        print(f"\nOptimal route to {hospital_name} from {location_name}:")
        print(f"- Distance: {route_distance:.2f} km")
        print(f"- Estimated travel time: {route_duration:.1f} minutes")

        # Create the map for the current location
        route_map = folium.Map(location=user_location, zoom_start=13)

        # Add user marker
        folium.Marker(
            location=user_location,
            popup=f"{location_name} (User Location)",
            icon=folium.Icon(color='blue')
        ).add_to(route_map)

        # Add hospital marker
        folium.Marker(
            location=hospital_coords,
            popup=f"{hospital_name} ({route_distance:.2f} km, {route_duration:.1f} min)",
            icon=folium.Icon(color='red')
        ).add_to(route_map)

        # Add the actual route (polyline of roads)
        folium.GeoJson(route_geometry, name='route').add_to(route_map)

        # Display the map inline
        display(route_map)

# Visualize routes for all 5 locations
visualize_multiple_routes(user_locations, hospital_locations)


Optimal route to THE ADDICTION INSTITUTE OF NEW YORK from Times Square, NYC:
- Distance: 1.99 km
- Estimated travel time: 4.0 minutes



Optimal route to MOUNT SINAI HOSPITAL from Central Park, NYC:
- Distance: 2.43 km
- Estimated travel time: 5.2 minutes



Optimal route to CALIFORNIA PACIFIC MED CTR-CALIFORNIA WEST from Golden Gate Bridge, SF:
- Distance: 9.77 km
- Estimated travel time: 13.1 minutes



Optimal route to GEORGE WASHINGTON UNIVERSITY HOSPITAL from White House, DC:
- Distance: 1.70 km
- Estimated travel time: 2.5 minutes



Optimal route to CEDAR-SINAI MARINA DEL REY HOSPITAL from LAX Airport, LA:
- Distance: 9.05 km
- Estimated travel time: 16.8 minutes
